In [0]:
!pip install -r /Workspace/Users/gengisgutierrezc@pacifico.com.pe/pacifico-metadata-ai/requirements.txt

In [0]:
import os
import time
import mlflow
from mlflow.models import infer_signature
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()
from agent import MetadataGovernanceAgent

In [0]:
mlflow.set_registry_uri("databricks-uc")
# Define tu catálogo y esquema
catalog_name = "iacg_chatbot_poc_desa"  # <-- Cambia esto si usas otro catálogo
schema_name = "coe_knowledge"  # <-- Cambia esto si usas otro esquema
model_name = f"{catalog_name}.{schema_name}.metabuilder_agent"
endpoint_name = "metabuilder-endpoint"

print(f"Model Name: {model_name}")
print(f"Endpoint Name: {endpoint_name}")

In [0]:
import mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, DataType, Object, Property, Array, AnyType

# Cada mensaje: {role, content}
message_schema = Object(properties=[
    Property("role",    DataType.string, required=True),
    Property("content", DataType.string, required=True),
])

# human_feedback: TODO opcional
human_feedback_schema = Object(properties=[
    Property("general_observations", DataType.string, required=False),
    Property("edited_table_comment", DataType.string, required=False),
    Property("edited_columns",       AnyType(),       required=False),
])

# custom_inputs: thread_id obligatorio, human_feedback opcional
custom_inputs_schema = Object(properties=[
    Property("thread_id",      DataType.string,       required=True),
    Property("human_feedback", human_feedback_schema, required=False), 
])

# Schema raíz de entrada
input_schema = Schema([
    ColSpec(name="messages",      type=Array(message_schema),  required=True),
    ColSpec(name="custom_inputs", type=custom_inputs_schema,   required=True),
])

# Schema de salida (string libre)
output_schema = Schema([ColSpec(type=DataType.string)])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# ─────────────────────────────────────────────────────────────
# 2. input_example (puedes dejarlo con o sin human_feedback)
# ─────────────────────────────────────────────────────────────
input_example = {
    "messages": [{"role": "user", "content": "main.default.my_table"}],
    "custom_inputs": {
        "thread_id": "12345"
        # human_feedback ya no es obligatorio
    },
}

# ─────────────────────────────────────────────────────────────
# 3. Log del modelo
# ─────────────────────────────────────────────────────────────
with mlflow.start_run() as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent_model",
        python_model="agent.py",
        code_path=[
            "agent.py", "graph", "state", "nodes",
            "prompts", "tools", "config", "context", "routers"
        ],
        pip_requirements="requirements.txt",
        input_example=input_example,
        signature=signature,   # 👈 firma explícita, ya no inferida
    )

print(f"Modelo registrado localmente en: {model_info.model_uri}")

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri, name=model_name
)
print(f"✅ Modelo {model_name} versión {registered_model.version} registrado en Unity Catalog.")

In [0]:
from databricks import agents

print(f"Desplegando el Agente de Gobierno en el Endpoint: {endpoint_name}...")

# databricks.agents.deploy() orquesta el Serving + Review App + Agent Evaluation
try:
    deployment_info = agents.deploy(
        model_name=model_name,
        model_version=registered_model.version,
        endpoint_name=endpoint_name,
        scale_to_zero=True,
        # Inyección de Secretos estilo "Agent Framework"
        environment_vars={
            "DATABRICKS_HOST": "{{secrets/metadatos_scope/db_host}}",
            "DATABRICKS_TOKEN": "{{secrets/metadatos_scope/db_token}}",
            "DATABRICKS_SQL_WAREHOUSE_ID": "{{secrets/metadatos_scope/db_warehouse_id}}"
        }
    )
    print(f"✅ Agente '{endpoint_name}' desplegado exitosamente.")
except Exception as e:
    raise e